# Limpieza de dataset - Malaria (P. Vivax)

Issue: [#7](https://github.com/Jacogz/DL-QPI/issues/7)

Objetivo: Aplicar pipeline de limpieza a las imágenes de malaria (campo claro RGB) para reducir la brecha con el dominio de fase QPI.

## Pipeline acordado
1. RGB → escala de grises
2. Rescale para igualar el tamaño de las células
3. Recorte en segmentos 256x256
4. Eliminación de imágenes (con menos de 3 células, con células infectadas por malaria, o con cúmulos de células que dificulten segmentarlas después)
5. Aumentación de datos (flips y rotaciones aleatorias)


## 1. Configuración y carga de datos

In [ ]:
import os
import json
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2

random.seed(123)
np.random.seed(123)

# Detectar raíz del repositorio
cwd = Path.cwd()
REPO_ROOT = cwd
while REPO_ROOT != REPO_ROOT.parent:
    if REPO_ROOT.name == 'DL-QPI' or (REPO_ROOT / 'pyproject.toml').exists() or (REPO_ROOT / '.git').exists():
        break
    REPO_ROOT = REPO_ROOT.parent

CANDIDATE_DATA_DIRS = [
    REPO_ROOT / 'data' / 'raw' / 'malaria',
    REPO_ROOT / 'data' / 'raw' / 'p-vivax-malaria-infected-human-blood-smears',
    Path('/content/data/p-vivax-malaria-infected-human-blood-smears'),
    Path('/kaggle/input/p-vivax-malaria-infected-human-blood-smears'),
    Path('/content/data'),
]

DATA_DIR = None
for d in CANDIDATE_DATA_DIRS:
    if d.exists():
        if (d / 'malaria').is_dir():
            DATA_DIR = d / 'malaria'
        else:
            DATA_DIR = d
        break

OUT_DIR = REPO_ROOT / 'data' / 'processed' / 'malaria'
OUT_DIR.mkdir(parents=True, exist_ok=True)

if DATA_DIR is None:
    print('No se encontró el dataset de malaria. Coloca los archivos en:')
    for d in CANDIDATE_DATA_DIRS:
        print(f' - {d}')
    raise FileNotFoundError('Dataset de malaria no encontrado')

print(f'REPO_ROOT : {REPO_ROOT}')
print(f'DATA_DIR  : {DATA_DIR}')
print(f'Salida    : {OUT_DIR.resolve()}')

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}
all_files = [f for f in Path(DATA_DIR).rglob('*') if f.is_file()]
image_files = sorted([f for f in all_files if f.suffix.lower() in IMAGE_EXTS])
json_files = sorted([f for f in all_files if f.suffix.lower() == '.json'])

print(f'Imágenes: {len(image_files)}')
print(f'JSON: {[j.name for j in json_files]}')

## 2. Cargar anotaciones y métricas por imagen

In [ ]:
annotation_rows = []
for jf in json_files:
    with open(jf) as f:
        data = json.load(f)
    for entry in data:
        image_path = entry['image']['pathname']
        img_name = Path(image_path).name
        for obj in entry['objects']:
            bbox = obj.get('bounding_box', {})
            annotation_rows.append({
                'split': jf.stem,
                'image_name': img_name,
                'category': obj['category'],
                'bbox_min_r': bbox.get('minimum', {}).get('r'),
                'bbox_min_c': bbox.get('minimum', {}).get('c'),
                'bbox_max_r': bbox.get('maximum', {}).get('r'),
                'bbox_max_c': bbox.get('maximum', {}).get('c'),
            })

ann_df = pd.DataFrame(annotation_rows)
print(f'Total anotaciones: {len(ann_df)}')
display(ann_df.head())

In [ ]:
NON_ERYTHROCYTE = {'leukocyte', 'difficult'}
INFECTED_CATS = {'trophozoite', 'ring', 'schizont', 'gametocyte'}

img_stats = []
for img_name, g in ann_df.groupby('image_name'):
    cats = g['category'].tolist()
    erythro = [c for c in cats if c not in NON_ERYTHROCYTE]
    n_cells = len(erythro)
    n_infected = sum(1 for c in erythro if c in INFECTED_CATS)
    img_stats.append({
        'image_name': img_name,
        'n_cells': n_cells,
        'n_infected': n_infected,
        'has_infected': n_infected > 0,
    })

img_stats_df = pd.DataFrame(img_stats)
print(img_stats_df.describe())
display(img_stats_df.head())

## 3. Funciones de preprocesamiento
- RGB → escala de grises
- Rescale para igualar tamaño de células (aproximado por bounding boxes)
- Recorte 256x256

In [ ]:
def rgb_to_gray(img):
    if isinstance(img, Image.Image):
        return img.convert('L')
    arr = np.array(img)
    if arr.ndim == 3:
        return Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY))
    return Image.fromarray(arr)


def estimate_cell_size_from_bboxes(df_img, img_name):
    rows = df_img[(df_img['image_name'] == img_name) & (~df_img['category'].isin(NON_ERYTHROCYTE))]
    if len(rows) == 0:
        return None
    sizes = []
    for _, r in rows.iterrows():
        if pd.notna(r['bbox_max_r']) and pd.notna(r['bbox_min_r']) and pd.notna(r['bbox_max_c']) and pd.notna(r['bbox_min_c']):
            h = r['bbox_max_r'] - r['bbox_min_r']
            w = r['bbox_max_c'] - r['bbox_min_c']
            sizes.append((h + w) / 2.0)
    return np.median(sizes) if sizes else None


def rescale_to_target_cell_size(img, current_cell_size, target_cell_size=80.0):
    if current_cell_size is None or current_cell_size <= 0:
        return img
    scale = target_cell_size / current_cell_size
    scale = float(np.clip(scale, 0.5, 2.0))
    w, h = img.size
    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))
    return img.resize((new_w, new_h), Image.LANCZOS)


def crop_to_patches(img, patch_size=256, stride=None):
    if stride is None:
        stride = patch_size
    arr = np.array(img)
    h, w = arr.shape[:2]
    patches = []
    for i in range(0, h - patch_size + 1, stride):
        for j in range(0, w - patch_size + 1, stride):
            patches.append(arr[i:i+patch_size, j:j+patch_size])
    return patches

## 4. Criterios de eliminación y conteo
- Menos de 3 células
- Con células infectadas por malaria
- Con cúmulos de células que dificulten segmentarlas después

In [ ]:
def has_dense_clusters(df_img, img_name):
    rows = df_img[(df_img['image_name'] == img_name) & (~df_img['category'].isin(NON_ERYTHROCYTE))]
    if len(rows) < 5:
        return False
    # Heurística: muchas células -> mayor probabilidad de cúmulos
    if len(rows) > 15:
        return True
    return False


removal_log = defaultdict(int)
kept_images = []

for img_path in image_files:
    img_name = img_path.name
    stats = img_stats_df[img_stats_df['image_name'] == img_name]
    if len(stats) == 0:
        removal_log['sin_anotaciones'] += 1
        continue
    s = stats.iloc[0]
    if s['n_cells'] < 3:
        removal_log['menos_de_3_celulas'] += 1
        continue
    if s['has_infected']:
        removal_log['con_celulas_infectadas'] += 1
        continue
    if has_dense_clusters(ann_df, img_name):
        removal_log['cumulos_dificiles'] += 1
        continue
    kept_images.append(img_path)

print('Conteo de imágenes eliminadas por criterio:')
for k, v in removal_log.items():
    print(f'  {k}: {v}')
print(f'\nImágenes mantenidas: {len(kept_images)}')

## 5. Aplicar pipeline y ejemplos antes/después

In [ ]:
n_show = min(3, len(kept_images))
if n_show > 0:
    fig, axes = plt.subplots(n_show, 4, figsize=(16, 4*n_show))
    if n_show == 1:
        axes = np.array([axes])

    for idx, img_path in enumerate(kept_images[:n_show]):
        img_orig = Image.open(img_path).convert('RGB')
        img_gray = rgb_to_gray(img_orig)
        cell_size = estimate_cell_size_from_bboxes(ann_df, img_path.name)
        img_rescaled = rescale_to_target_cell_size(img_gray, cell_size, target_cell_size=80.0)
        patches = crop_to_patches(img_rescaled, patch_size=256)
        patch_show = patches[0] if patches else np.array(img_rescaled)

        axes[idx, 0].imshow(img_orig); axes[idx, 0].set_title('Original (RGB)')
        axes[idx, 1].imshow(img_gray, cmap='gray'); axes[idx, 1].set_title('Escala de grises')
        axes[idx, 2].imshow(img_rescaled, cmap='gray'); axes[idx, 2].set_title(f'Rescale (cell~{cell_size if cell_size else 0:.1f}px)')
        axes[idx, 3].imshow(patch_show, cmap='gray'); axes[idx, 3].set_title('Parche 256x256')

    for ax in axes.ravel():
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No hay imágenes mantenidas para mostrar.')

## 6. Aumentación de datos (flips y rotaciones aleatorias)

In [ ]:
def augment_image(img_arr):
    augmented = []
    augmented.append(img_arr)
    augmented.append(np.fliplr(img_arr))
    augmented.append(np.flipud(img_arr))
    augmented.append(np.rot90(img_arr, 1))
    augmented.append(np.rot90(img_arr, 2))
    augmented.append(np.rot90(img_arr, 3))
    return augmented

In [ ]:
count_saved = 0
for img_path in kept_images:
    try:
        img_orig = Image.open(img_path).convert('RGB')
        img_gray = rgb_to_gray(img_orig)
        cell_size = estimate_cell_size_from_bboxes(ann_df, img_path.name)
        img_rescaled = rescale_to_target_cell_size(img_gray, cell_size, target_cell_size=80.0)
        patches = crop_to_patches(img_rescaled, patch_size=256)
        for i, p in enumerate(patches):
            if p.ndim == 3:
                p_gray = cv2.cvtColor(p, cv2.COLOR_RGB2GRAY)
            else:
                p_gray = p
            aug_list = augment_image(p_gray)
            for j, aug in enumerate(aug_list):
                out_name = f"{img_path.stem}_p{i:03d}_aug{j:02d}.png"
                Image.fromarray(aug.astype(np.uint8)).save(OUT_DIR / out_name)
                count_saved += 1
    except Exception as e:
        print(f'Error procesando {img_path.name}: {e}')

print(f'Imágenes guardadas (parches + aumentados): {count_saved}')
print(f'Directorio: {OUT_DIR.resolve()}')